# DHI adoption and engagement descriptives

Describes when the main analysis sample adopted the DHI and how they used it afterwards: adoption timing, engagement and retention over time, deductible (franchise) changes around adoption, use of reward modules and health sessions, timing of chronic diagnoses (PCGs), and reward points paid out after adoption.

In [ ]:
import sys
from pathlib import Path
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
from matplotlib.ticker import PercentFormatter

# Import shared database configuration
ROOT_DIR = Path.cwd()
REPO_ROOT = ROOT_DIR.parent
sys.path.insert(0, str(REPO_ROOT))
from data_preparation.config import connect_db

# Paths
OUT_DIR = ROOT_DIR / "results" / "descriptives"
DB_NAME = REPO_ROOT / "insurance.duckdb"
unique_ids_main_analysis = REPO_ROOT / "unique_ids_final.csv"

OUT_DIR.mkdir(parents=True, exist_ok=True)

# Load analysis cohort
main_analysis_ids = pd.read_csv(unique_ids_main_analysis)

con = connect_db(str(DB_NAME))

# Analysis-specific DuckDB settings
TEMP_DIR = ROOT_DIR / "tmp" / "duckdb"
TEMP_DIR.mkdir(parents=True, exist_ok=True)

con.execute("SET memory_limit = '48GB'")
con.execute("SET threads = 8")
con.execute(f"SET temp_directory = '{TEMP_DIR.as_posix()}'")
con.execute("SET max_temp_directory_size = '14GiB'")
con.execute("SET preserve_insertion_order = false")

## Sanity check: table + columns

In [ ]:
con.execute("DESCRIBE a365_transaction_check").df()


In [ ]:
con.execute("SHOW ALL TABLES").df()

### Formatting

In [ ]:
# ---- Lancet-style number formatting ---------------------------------
# Decimal point -> centred dot
# Thousands:
#   1000–9999  -> 7753
#   >=10 000   -> 10 234 (thin space)

THIN_SPACE = "\u2009"
MID_DOT = "\u00b7"

def fmt_int_lancet(x):
    """Lancet-style integer formatting."""
    x = int(round(x))
    sign = "-" if x < 0 else ""
    s = str(abs(x))

    if len(s) <= 4:
        return sign + s

    # group digits from the right using thin spaces
    groups = []
    while s:
        groups.append(s[-3:])
        s = s[:-3]

    return sign + THIN_SPACE.join(reversed(groups))


def fmt_dec_lancet(x, decimals=1):
    """Lancet-style decimal formatting with centred decimal point."""
    return f"{x:.{decimals}f}".replace(".", MID_DOT)


def fmt_pct_lancet(x, decimals=1):
    """Lancet-style percentage formatting."""
    return f"{fmt_dec_lancet(x, decimals)}%"


## 1. People who *acquired* a transaction in 2024
(first-ever transaction date falls in 2024 — i.e. 2024 adoption cohort)


In [ ]:
acquired_2024 = con.execute("""
    WITH first_txn AS (
        SELECT
            CSS_MARKETING_ID,
            MIN(TRANSACTION_DATETIME) AS first_transaction_date
        FROM a365_transaction_check
        GROUP BY CSS_MARKETING_ID
    )
    SELECT COUNT(DISTINCT CSS_MARKETING_ID) AS n_acquired_2024
    FROM first_txn
    WHERE EXTRACT(YEAR FROM first_transaction_date) = 2024
""").df()

acquired_2024


## 2. (Alternative) People with *any* transaction dated in 2024
Includes people who adopted earlier but also transacted
again during 2024.


In [ ]:
any_txn_2024 = con.execute("""
    SELECT COUNT(DISTINCT CSS_MARKETING_ID) AS n_any_transaction_2024
    FROM a365_transaction_check
    WHERE EXTRACT(YEAR FROM TRANSACTION_DATETIME) = 2024
""").df()

any_txn_2024


## Monthly breakdown (2024 acquisitions), for a sanity-check trend


In [ ]:
monthly_acquired_2024 = con.execute("""
    WITH first_txn AS (
        SELECT
            CSS_MARKETING_ID,
            MIN(TRANSACTION_DATETIME) AS first_transaction_date
        FROM a365_transaction_check
        GROUP BY CSS_MARKETING_ID
    )
    SELECT
        DATE_TRUNC('month', first_transaction_date) AS acquisition_month,
        COUNT(DISTINCT CSS_MARKETING_ID) AS n_acquired
    FROM first_txn
    WHERE EXTRACT(YEAR FROM first_transaction_date) = 2024
    GROUP BY 1
    ORDER BY 1
""").df()

monthly_acquired_2024


## People acquiring any points per year 2020-2024

In [ ]:
yearly_unique = con.execute("""
    SELECT
        EXTRACT(YEAR FROM TRANSACTION_DATETIME) AS year,
        COUNT(DISTINCT CSS_MARKETING_ID) AS n_unique_people
    FROM a365_transaction_check
    WHERE EXTRACT(YEAR FROM TRANSACTION_DATETIME) BETWEEN 2020 AND 2024
    GROUP BY 1
    ORDER BY 1
""").df()

print(yearly_unique)

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(yearly_unique["year"].astype(str), yearly_unique["n_unique_people"], color="#4C72B0")
ax.set_xlabel("Year", fontsize=14)
ax.set_ylabel("Unique people with a transaction",fontsize=14)

for i, v in enumerate(yearly_unique["n_unique_people"]):
    ax.text(i, v, f"{v:,}", ha="center", va="bottom")

plt.tight_layout()
plt.savefig("yearly_unique_transactors.png", dpi=200)
plt.show()

## Adoption timing

In [ ]:
def compute_monthly_adoption(
    con,
    use_main_ids: bool = False,
    main_ids_df: pd.DataFrame = None,
    main_ids_col: str = "eth_id",
    person_table: str = "persons",
    person_id_col: str = "eth_id",
    contract_table: str = "contracts_obligatory",
    contract_id_col: str = "eth_id",
    claim_tables: tuple = ("outpatient", "inpatient"),
    claim_id_col: str = "eth_id",
    credit_value: str = "Credit",
    start: str = "2020-01-01",
    end_exclusive: str = "2025-01-01",
    out_dir: str = None,
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Monthly count of new adopters (first credit transaction).

    use_main_ids=True  -> sample = ids in main_ids_df (main analysis sample)
    use_main_ids=False -> sample = people who
                          (1) have >=1 credit transaction with positive value,
                          (2) are linked to the insurance person table,
                          (3) have >=1 mandatory insurance contract ever,
                          (4) have >=1 claim in outpatient or inpatient.
    """
    # 1. Sample ids -> temp table
    if use_main_ids:
        if main_ids_df is None:
            raise ValueError("main_ids_df required when use_main_ids=True")
        ids = main_ids_df[[main_ids_col]].astype(str).drop_duplicates()
        ids.columns = ["sample_id"]
        con.register("main_ids_view", ids)
        con.execute("CREATE OR REPLACE TEMP TABLE sample_ids AS SELECT sample_id FROM main_ids_view")
        label = "main_ids"
    else:
        con.execute(f"""
            CREATE OR REPLACE TEMP TABLE sample_step1 AS
            SELECT DISTINCT CAST(CSS_MARKETING_ID AS VARCHAR) AS sample_id
            FROM a365_transaction_check
            WHERE CSS_MARKETING_ID IS NOT NULL
              AND UPPER(TRIM(TRANSACTION_TYPE)) = UPPER('{credit_value}')
              AND TRANSACTION_VALUE > 0
        """)
        con.execute(f"""
            CREATE OR REPLACE TEMP TABLE sample_step2 AS
            SELECT sample_id FROM sample_step1
            WHERE sample_id IN (SELECT DISTINCT CAST({person_id_col} AS VARCHAR) FROM {person_table})
        """)
        con.execute(f"""
            CREATE OR REPLACE TEMP TABLE sample_step3 AS
            SELECT sample_id FROM sample_step2
            WHERE sample_id IN (SELECT DISTINCT CAST({contract_id_col} AS VARCHAR) FROM {contract_table})
        """)
        claims_union = " UNION ".join(
            f"SELECT DISTINCT CAST({claim_id_col} AS VARCHAR) AS id FROM {tbl}" for tbl in claim_tables
        )
        con.execute(f"""
            CREATE OR REPLACE TEMP TABLE sample_ids AS
            SELECT sample_id FROM sample_step3
            WHERE sample_id IN ({claims_union})
        """)
        label = "a365_linked"

        if verbose:
            funnel = con.execute("""
                SELECT
                    (SELECT COUNT(*) FROM sample_step1) AS n_credit_gt0,
                    (SELECT COUNT(*) FROM sample_step2) AS n_linked,
                    (SELECT COUNT(*) FROM sample_step3) AS n_mandatory_contract,
                    (SELECT COUNT(*) FROM sample_ids)   AS n_any_claim
            """).df()
            print("[a365_linked funnel]")
            print(funnel.to_string(index=False))

    # 2. First credit per person -> temp table (computed once)
    con.execute(f"""
        CREATE OR REPLACE TEMP TABLE first_credit AS
        SELECT CAST(t.CSS_MARKETING_ID AS VARCHAR) AS id,
               MIN(t.TRANSACTION_DATETIME)         AS first_credit_dt
        FROM a365_transaction_check t
        WHERE UPPER(TRIM(t.TRANSACTION_TYPE)) = UPPER('{credit_value}')
          AND CAST(t.CSS_MARKETING_ID AS VARCHAR) IN (SELECT sample_id FROM sample_ids)
        GROUP BY 1
    """)

    # 3. Monthly counts, month grid in pandas
    monthly = con.execute(f"""
        SELECT DATE_TRUNC('month', first_credit_dt)::DATE AS adoption_month,
               COUNT(*) AS n_adopters
        FROM first_credit
        WHERE first_credit_dt >= TIMESTAMP '{start}'
          AND first_credit_dt <  TIMESTAMP '{end_exclusive}'
        GROUP BY 1
    """).df()
    monthly["adoption_month"] = pd.to_datetime(monthly["adoption_month"])

    grid = pd.DataFrame({"adoption_month": pd.date_range(
        start, pd.Timestamp(end_exclusive) - pd.offsets.MonthBegin(1), freq="MS")})
    df = grid.merge(monthly, on="adoption_month", how="left")
    df["n_adopters"] = df["n_adopters"].fillna(0).astype(int)
    df["cumulative_adopters"] = df["n_adopters"].cumsum()
    df.attrs["sample"] = label

    # 4. Diagnostics + consistency check
    diag = con.execute(f"""
        SELECT
            (SELECT COUNT(*) FROM sample_ids) AS n_sample,
            COUNT(*) AS n_with_credit,
            SUM(CASE WHEN first_credit_dt <  TIMESTAMP '{start}' THEN 1 ELSE 0 END)         AS n_before_window,
            SUM(CASE WHEN first_credit_dt >= TIMESTAMP '{end_exclusive}' THEN 1 ELSE 0 END) AS n_after_window
        FROM first_credit
    """).df()
    n_in_window = int(diag.n_with_credit[0] - diag.n_before_window[0] - diag.n_after_window[0])
    assert df["n_adopters"].sum() == n_in_window, (
        f"Mismatch: monthly sum {df['n_adopters'].sum()} vs {n_in_window} in window")

    if verbose:
        print(f"[{label}]")
        print(diag.to_string(index=False))
        print(f"Adopters in window: {df['n_adopters'].sum():,}")

    if out_dir is not None:
        df.to_csv(Path(out_dir) / f"adoption_by_month_{label}.csv", index=False)

    return df

In [ ]:
adopt_main = compute_monthly_adoption(con, use_main_ids=True,
                                      main_ids_df=main_analysis_ids, out_dir=OUT_DIR)

fig, ax = plt.subplots(figsize=(12, 4.5))

ax.plot(adopt_main["adoption_month"], adopt_main["n_adopters"],
        color="#D1495B", lw=1.8, marker="o", ms=3,
        label=f"Main analysis sample (n = {adopt_main['n_adopters'].sum():,})")

for yr in range(2021, 2025):
    ax.axvline(pd.Timestamp(f"{yr}-01-01"), color="grey", lw=0.6, ls="--", alpha=0.5)

ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.xaxis.set_minor_locator(mdates.MonthLocator())          # tick every month, no label
ax.xaxis.set_minor_formatter(plt.NullFormatter())
ax.set_xlim(pd.Timestamp("2019-12-31"), pd.Timestamp("2024-12-31"))

# Marketing pushes: green arrows pointing at the main-sample line
pushes = [
    (pd.Timestamp("2021-04-16"), ["2021-04-01", "2021-05-01"], "Marketing push\n(Apr/May 2021)", 0.30, -60),
    (pd.Timestamp("2021-09-01"), ["2021-09-01"],               "Marketing push\n(Sep 2021)",     0.10,  60),
]
y_max = adopt_main["n_adopters"].max()
for x, months, txt, dy, dx_days in pushes:
    y = adopt_main.loc[adopt_main["adoption_month"].isin(pd.to_datetime(months)), "n_adopters"].max()
    ax.annotate(txt, xy=(x, y),
                xytext=(x + pd.Timedelta(days=dx_days), y + dy * y_max),
                ha="center", va="bottom", fontsize=9, color="#2E8B57",
                arrowprops=dict(arrowstyle="-|>", color="#2E8B57", lw=1.5))

ax.set_ylim(bottom=0, top=y_max * 1.5)   # more headroom for the higher label

ax.set_ylabel("New adopters (first credit transaction)")
ax.set_xlabel("Calendar month")
ax.set_title("Adoption of the DHI by month, 2020–2024")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.3)
ax.legend(frameon=False, loc="upper right")

plt.tight_layout()
fig.savefig(Path(OUT_DIR) / "adoption_by_month_main.png", dpi=300, bbox_inches="tight")
fig.savefig(Path(OUT_DIR) / "adoption_by_month_main.pdf", bbox_inches="tight")
plt.show()

## Engagement over time

In [ ]:
# 1. Report engagement trajectory (i.e., how long do ppl stay active in the DHI - how long do they keep earning points) - figure
# =====================================================================
#    Engagement trajectory: how long do adopters keep earning points?
#    Kaplan–Meier of time from adoption to last point-earning day.
#    Churn = no point-earning day in the final CHURN_GAP_DAYS of data;
#    everyone else is right-censored at DATA_END.
# =====================================================================

CHURN_GAP_DAYS = 90
MONTH_DAYS = 30.4375

data_max = pd.Timestamp(con.execute(
    "SELECT MAX(TRANSACTION_DATETIME) FROM a365_transaction_check").fetchone()[0])
DATA_END = min(data_max.normalize(), pd.Timestamp("2024-12-31"))
print(f"Data end used: {DATA_END.date()}  (table max: {data_max})")

# Sample = main analysis ids (rebuilt: sample_ids gets overwritten by each compute_monthly_adoption call)
ids = main_analysis_ids[["eth_id"]].astype(str).drop_duplicates().rename(columns={"eth_id": "id"})
con.register("eng_ids_view", ids)
con.execute("CREATE OR REPLACE TEMP TABLE eng_ids AS SELECT id FROM eng_ids_view")

# Adoption = first CREDIT transaction 
con.execute(f"""
    CREATE OR REPLACE TEMP TABLE eng_adopt AS
    SELECT CAST(CSS_MARKETING_ID AS VARCHAR)        AS id,
           CAST(MIN(TRANSACTION_DATETIME) AS DATE)  AS adopt_d
    FROM a365_transaction_check
    WHERE UPPER(TRIM(TRANSACTION_TYPE)) = 'CREDIT'
      AND CAST(CSS_MARKETING_ID AS VARCHAR) IN (SELECT id FROM eng_ids)
    GROUP BY 1
    HAVING MIN(TRANSACTION_DATETIME) < TIMESTAMP '{DATA_END + pd.Timedelta(days=1)}'
""")

# Point-earning days = distinct person-days with a positive CREDIT, from adoption to DATA_END
con.execute(f"""
    CREATE OR REPLACE TEMP TABLE eng_earn AS
    SELECT DISTINCT CAST(t.CSS_MARKETING_ID AS VARCHAR) AS id,
           CAST(t.TRANSACTION_DATETIME AS DATE)         AS d
    FROM a365_transaction_check t
    JOIN eng_adopt a ON a.id = CAST(t.CSS_MARKETING_ID AS VARCHAR)
    WHERE UPPER(TRIM(t.TRANSACTION_TYPE)) = 'CREDIT'
      AND t.TRANSACTION_VALUE > 0
      AND CAST(t.TRANSACTION_DATETIME AS DATE) BETWEEN a.adopt_d AND DATE '{DATA_END.date()}'
""")

person = con.execute("""
    SELECT a.id, a.adopt_d, MAX(e.d) AS last_d, COUNT(e.d) AS n_earn_days
    FROM eng_adopt a LEFT JOIN eng_earn e USING (id)
    GROUP BY 1, 2
""").df()
for c in ["adopt_d", "last_d"]:
    person[c] = pd.to_datetime(person[c])
person["last_d"]  = person["last_d"].fillna(person["adopt_d"])       # adopters with no positive credit
person["churned"] = person["last_d"] < DATA_END - pd.Timedelta(days=CHURN_GAP_DAYS)
person["t_end"]   = person["last_d"].where(person["churned"], DATA_END)
person["months"]  = (person["t_end"] - person["adopt_d"]).dt.days / MONTH_DAYS
person["cohort"]  = person["adopt_d"].dt.year
print(f"Adopters: {len(person):,} | churned: {person['churned'].mean():.1%} | censored: {(~person['churned']).mean():.1%}")

def kaplan_meier(t, event):
    d = pd.DataFrame({"t": t.values, "e": event.astype(int).values})
    tab = d.groupby("t")["e"].agg(events="sum", n="size").sort_index()
    tab["at_risk"] = tab["n"][::-1].cumsum()[::-1]
    tab["S"] = (1 - tab["events"] / tab["at_risk"]).cumprod()
    start = pd.DataFrame({"events": [0], "n": [0], "at_risk": [len(d)], "S": [1.0]}, index=[0.0])
    return pd.concat([start, tab])

def km_median(km):
    below = km.index[km["S"] <= 0.5]
    return below.min() if len(below) else np.nan

km_all = kaplan_meier(person["months"], person["churned"])
med = km_median(km_all)

# Summary table at fixed horizons
checkpoints = [3, 6, 12, 24, 36, 48]
km_summary = pd.DataFrame({
    "months": checkpoints,
    "share_still_active": [km_all.loc[km_all.index <= m, "S"].iloc[-1] for m in checkpoints],
    "n_at_risk": [(person["months"] >= m).sum() for m in checkpoints],
})
print(f"Median time to last point-earning: {med:.1f} months" if pd.notna(med) else "Median not reached")
print(km_summary.to_string(index=False))

# Figure
fig, ax = plt.subplots(figsize=(9, 5))
ax.step(km_all.index, km_all["S"], where="post", color="black", lw=2.2,
        label=f"All adopters (n = {len(person):,})")
cohorts = sorted(person["cohort"].unique())
cmap = plt.get_cmap("viridis", len(cohorts))
for i, yr in enumerate(cohorts):
    g = person[person["cohort"] == yr]
    km_c = kaplan_meier(g["months"], g["churned"])
    ax.step(km_c.index, km_c["S"], where="post", lw=1.2, color=cmap(i), alpha=0.9,
            label=f"Adopted {yr} (n = {len(g):,})")
if pd.notna(med):
    ax.axhline(0.5, color="grey", lw=0.6, ls="--")
    ax.axvline(med, color="grey", lw=0.6, ls="--")
    ax.annotate(f"Median = {med:.1f} months", xy=(med, 0.5), xytext=(med + 2, 0.58), fontsize=9)

ax.set_xlim(0, person["months"].max())
ax.set_ylim(0, 1.02)
ax.yaxis.set_major_formatter(PercentFormatter(1))
ax.set_xlabel("Months since adoption")
ax.set_ylabel("Share still earning points")
ax.set_title("Engagement retention after adoption (Kaplan–Meier)")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.3)
ax.legend(frameon=False, fontsize=8, loc="lower right")
plt.tight_layout()
fig.savefig(Path(OUT_DIR) / "engagement_km_main.png", dpi=300, bbox_inches="tight")
fig.savefig(Path(OUT_DIR) / "engagement_km_main.pdf", bbox_inches="tight")
plt.show()

km_summary.to_csv(Path(OUT_DIR) / "engagement_km_summary_main.csv", index=False)

In [ ]:
# 2. todo: figure with the share of adopters with any point-earning day in each quarter after adoption and the median number of active months
# =====================================================================
# 2) Share of adopters with >=1 point-earning day in each quarter after
#    adoption + median (IQR) cumulative number of active months.
#    Requires block 1 (eng_adopt, eng_earn, DATA_END).
#    Quarter k = calendar months [3(k-1), 3k-1] counted from the adoption month.
#    A quarter's denominator = adopters whose quarter is fully observed by DATA_END.
# =====================================================================
MIN_N = 500   # drop quarters with too few observable adopters

pm = con.execute("""
    SELECT e.id,
           DATE_DIFF('month', DATE_TRUNC('month', a.adopt_d), DATE_TRUNC('month', e.d)) AS rel_m
    FROM eng_earn e JOIN eng_adopt a USING (id)
    GROUP BY 1, 2
""").df()
pm["q"] = pm["rel_m"] // 3

adopters = con.execute("SELECT id, adopt_d FROM eng_adopt").df()
adopters["adopt_d"] = pd.to_datetime(adopters["adopt_d"])
last_full_m = ((DATA_END.year - adopters["adopt_d"].dt.year) * 12
               + (DATA_END.month - adopters["adopt_d"].dt.month)
               - (0 if DATA_END.is_month_end else 1))
adopters["max_q"] = (last_full_m - 2) // 3          # last fully observed quarter (0-based)
max_q = adopters.set_index("id")["max_q"]

n_q = int(max_q.max()) + 1
months_in_q = (pm.groupby(["id", "q"]).size()
                 .unstack(fill_value=0)
                 .reindex(index=max_q.index, columns=range(n_q), fill_value=0))
cum_months = months_in_q.cumsum(axis=1)

rows = []
for k in range(n_q):
    elig = max_q.values >= k
    n = int(elig.sum())
    if n < MIN_N:
        continue
    active = months_in_q.values[elig, k] > 0
    c = cum_months.values[elig, k]
    rows.append(dict(quarter=k + 1, n_eligible=n, n_active=int(active.sum()),
                     share_active=active.mean(),
                     median_cum_active_months=np.median(c),
                     p25=np.percentile(c, 25), p75=np.percentile(c, 75)))
qdf = pd.DataFrame(rows)
print(qdf.to_string(index=False))

# Figure
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7.5), sharex=True,
                               gridspec_kw={"height_ratios": [1.1, 1]})

ax1.bar(qdf["quarter"], qdf["share_active"], color="#4C72B0")
for x, v in zip(qdf["quarter"], qdf["share_active"]):
    ax1.text(x, v + 0.01, f"{v:.0%}", ha="center", va="bottom", fontsize=7)
ax1.set_ylim(0, 1.08)
ax1.yaxis.set_major_formatter(PercentFormatter(1))
ax1.set_ylabel("Adopters with ≥1 point-earning day")
ax1.set_title("Engagement by quarter since adoption")
ax1.text(0.99, 0.95, "Q1 includes the adoption month", transform=ax1.transAxes,
         ha="right", va="top", fontsize=8, color="grey")

ax2.plot(qdf["quarter"], qdf["median_cum_active_months"], color="#D1495B", marker="o", ms=3,
         lw=1.8, label="Median")
ax2.fill_between(qdf["quarter"], qdf["p25"], qdf["p75"], color="#D1495B", alpha=0.2, label="IQR")
ax2.plot(qdf["quarter"], 3 * qdf["quarter"], color="grey", lw=0.8, ls="--", label="Maximum possible")
ax2.set_ylabel("Cumulative active months")
ax2.set_xlabel("Quarter since adoption")
ax2.legend(frameon=False, loc="upper left", fontsize=8)

for ax in (ax1, ax2):
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", alpha=0.3)
ax2.set_xticks(qdf["quarter"])

plt.tight_layout()
fig.savefig(Path(OUT_DIR) / "engagement_by_quarter_main.png", dpi=300, bbox_inches="tight")
fig.savefig(Path(OUT_DIR) / "engagement_by_quarter_main.pdf", bbox_inches="tight")
plt.show()

qdf.to_csv(Path(OUT_DIR) / "engagement_by_quarter_main.csv", index=False)

In [ ]:
cal = con.execute("""
    WITH act AS (SELECT DISTINCT id, DATE_TRUNC('quarter', d) AS cq FROM eng_earn),
         qs  AS (SELECT DISTINCT DATE_TRUNC('quarter', d) AS cq FROM eng_earn),
         grid AS (
             SELECT a.id, YEAR(a.adopt_d) AS cohort, qs.cq
             FROM eng_adopt a JOIN qs ON qs.cq > DATE_TRUNC('quarter', a.adopt_d)
         )
    SELECT grid.cohort, grid.cq, COUNT(*) AS n,
           AVG(CASE WHEN act.id IS NOT NULL THEN 1.0 ELSE 0 END) AS share
    FROM grid LEFT JOIN act ON act.id = grid.id AND act.cq = grid.cq
    GROUP BY 1, 2 ORDER BY 1, 2
""").df()
cal["cq"] = pd.to_datetime(cal["cq"])

fig, ax = plt.subplots(figsize=(10, 4.5))
for yr, g in cal.groupby("cohort"):
    ax.plot(g["cq"], g["share"], marker="o", ms=3, label=f"Adopted {yr}")
ax.yaxis.set_major_formatter(PercentFormatter(1))
ax.set_ylim(0, 1.05)
ax.set_ylabel("Share of prior adopters earning points")
ax.set_xlabel("Calendar quarter")
ax.legend(frameon=False, fontsize=8)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
MAX_Q = 12

# --- from block 2: activity by quarter since adoption ---
pm = con.execute("""
    SELECT e.id,
           DATE_DIFF('month', DATE_TRUNC('month', a.adopt_d), DATE_TRUNC('month', e.d)) AS rel_m
    FROM eng_earn e JOIN eng_adopt a USING (id)
    GROUP BY 1, 2
""").df()
pm["q"] = pm["rel_m"] // 3

adopters = con.execute("SELECT id, adopt_d FROM eng_adopt").df()
adopters["adopt_d"] = pd.to_datetime(adopters["adopt_d"])
last_full_m = ((DATA_END.year - adopters["adopt_d"].dt.year) * 12
               + (DATA_END.month - adopters["adopt_d"].dt.month)
               - (0 if DATA_END.is_month_end else 1))
adopters["max_q"] = (last_full_m - 2) // 3
max_q = adopters.set_index("id")["max_q"]

n_q = int(max_q.max()) + 1
months_in_q = (pm.groupby(["id", "q"]).size()
                 .unstack(fill_value=0)
                 .reindex(index=max_q.index, columns=range(n_q), fill_value=0))

# --- fig_df construction ---
A = months_in_q.values[:, :MAX_Q] > 0              # active in quarter k
cont = np.cumprod(A, axis=1).astype(bool)          # active in every quarter up to k

rows = []
for k in range(MAX_Q):
    elig = max_q.values >= k
    a_k, c_k = A[elig, k], cont[elig, k]
    rows.append(dict(quarter=k + 1, n=int(elig.sum()),
                     continuous=c_k.mean(),
                     returned=(a_k & ~c_k).mean(),
                     inactive=(~a_k).mean()))
fig_df = pd.DataFrame(rows)
fig_df["active"] = fig_df["continuous"] + fig_df["returned"]
print(fig_df.round(3).to_string(index=False))

states = [
    ("continuous", "#1F4E79", "Active in every quarter since adoption"),
    ("returned",   "#6FA8DC", "Active, after ≥1 inactive quarter"),
    ("inactive",   "#C8C8C8", "Inactive in this quarter"),
]
MIN_LABEL = 0.04   # don't label segments smaller than 4 pp

with plt.rc_context({"font.size": 13}):
    fig, ax = plt.subplots(figsize=(13, 6.2))
    x, bottom = fig_df["quarter"], np.zeros(len(fig_df))
    for col, color, lab in states:
        ax.bar(x, fig_df[col], bottom=bottom, color=color, width=0.78,
               edgecolor="white", linewidth=0.6, label=lab)
        bottom += fig_df[col].values

    for xi, a, c, r in zip(x, fig_df["active"], fig_df["continuous"], fig_df["returned"]):
        ax.text(xi, a + 0.012, f"{a:.0%}", ha="center", va="bottom",
                fontsize=11, fontweight="bold")                      # total active
        if xi > 1:
            ax.text(xi, c / 2, f"{c:.0%}", ha="center", va="center",
                    fontsize=11, color="white")                      # continuous
        if r >= MIN_LABEL:
            ax.text(xi, c + r / 2, f"{r:.0%}", ha="center", va="center",
                    fontsize=11, color="#0B2545")                    # returning

    ax.set_xticks(x)
    ax.set_xticklabels([f"Q{q}\n(n={n:,})" for q, n in zip(x, fig_df["n"])], fontsize=10)
    ax.set_xlabel("Quarter since adoption (event time)", fontsize=14, labelpad=8)
    ax.set_ylabel("Share of adopters", fontsize=14)
    ax.tick_params(axis="y", labelsize=12)
    ax.set_ylim(0, 1)
    ax.yaxis.set_major_formatter(PercentFormatter(1))
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False, fontsize=12, loc="upper center",
              bbox_to_anchor=(0.5, -0.2), ncol=3, handlelength=1.5, columnspacing=2)
    plt.tight_layout()
    fig.savefig(Path(OUT_DIR) / "engagement_states_by_quarter_main.png", dpi=300, bbox_inches="tight")
    fig.savefig(Path(OUT_DIR) / "engagement_states_by_quarter_main.pdf", bbox_inches="tight")
    plt.show()

# Transitions in deductible/franchise

In [ ]:
# =====================================================================
# Franchise per person-calendar-year (children's franchises -> NA)
# + df: franchise in calendar year before adoption vs. adoption year
# =====================================================================
cols = con.execute("DESCRIBE contracts_obligatory").df()["column_name"].tolist()
fr_candidates = [c for c in cols if "franchis" in c.lower()]
if not fr_candidates:
    raise ValueError(f"No franchise column found in contracts_obligatory: {cols} -> set FR_COL manually")
FR_COL = fr_candidates[0]
print("Franchise column:", FR_COL)

CHILD_FRANCHISES = (0, 100, 200, 400, 600)
MAX_YEAR = 2026  # cap for open-ended contracts

def to_date(c):
    return (f"COALESCE(TRY_CAST({c} AS DATE), TRY_CAST(TRY_CAST({c} AS TIMESTAMP) AS DATE), "
            f"CAST(TRY_STRPTIME({c}, '%d.%m.%Y') AS DATE), CAST(TRY_STRPTIME({c}, '%Y%m%d') AS DATE))")

ids = main_analysis_ids[["eth_id"]].astype(str).drop_duplicates().rename(columns={"eth_id": "id"})
con.register("fr_ids_view", ids)
con.execute("CREATE OR REPLACE TEMP TABLE fr_ids AS SELECT id FROM fr_ids_view")

# Adoption calendar year = year of first CREDIT transaction
con.execute("""
    CREATE OR REPLACE TEMP TABLE fr_adopt AS
    SELECT CAST(CSS_MARKETING_ID AS VARCHAR) AS id,
           YEAR(MIN(TRANSACTION_DATETIME))   AS adopt_year
    FROM a365_transaction_check
    WHERE UPPER(TRIM(TRANSACTION_TYPE)) = 'CREDIT'
      AND CAST(CSS_MARKETING_ID AS VARCHAR) IN (SELECT id FROM fr_ids)
    GROUP BY 1
""")

# One row per person-calendar-year
child_list = ", ".join(str(v) for v in CHILD_FRANCHISES)
con.execute(f"""
    CREATE OR REPLACE TEMP TABLE fr_py AS
    WITH c AS (
        SELECT CAST(eth_id AS VARCHAR)                                   AS id,
               {to_date('beginndatum')}                                  AS b,
               COALESCE({to_date('endedatum')}, DATE '{MAX_YEAR}-12-31') AS e,
               CASE WHEN {FR_COL} IN ({child_list}) THEN NULL ELSE {FR_COL} END AS franchise
        FROM contracts_obligatory
        WHERE CAST(eth_id AS VARCHAR) IN (SELECT id FROM fr_ids)
    ),
    yrs AS (
        SELECT id, franchise, UNNEST(range(YEAR(b), LEAST(YEAR(e), {MAX_YEAR}) + 1)) AS yr
        FROM c
        WHERE b IS NOT NULL AND e >= b
    )
    SELECT id, yr, ANY_VALUE(franchise) AS franchise
    FROM yrs
    GROUP BY 1, 2
""")

def franchise_pair(off_a, off_b, lab_a, lab_b):
    df = con.execute(f"""
        SELECT a.id AS eth_id, a.adopt_year,
               p.franchise AS franchise_{lab_a},
               q.franchise AS franchise_{lab_b}
        FROM fr_adopt a
        LEFT JOIN fr_py p ON p.id = a.id AND p.yr = a.adopt_year + ({off_a})
        LEFT JOIN fr_py q ON q.id = a.id AND q.yr = a.adopt_year + ({off_b})
        ORDER BY a.id
    """).df()
    both = df[f"franchise_{lab_a}"].notna() & df[f"franchise_{lab_b}"].notna()
    df["franchise_changed"] = (df[f"franchise_{lab_a}"] != df[f"franchise_{lab_b}"]).where(both)
    return df

franchise_prior_adopt = franchise_pair(-1, 0, "prior", "adopt")

print(f"n = {len(franchise_prior_adopt):,}")
print(franchise_prior_adopt[["franchise_prior", "franchise_adopt"]].isna().mean().rename("share_NA"))
print(pd.crosstab(franchise_prior_adopt["franchise_prior"], franchise_prior_adopt["franchise_adopt"], dropna=False))
franchise_prior_adopt.head()

In [ ]:
# =====================================================================
# df: franchise in adoption year vs. calendar year after adoption
# =====================================================================
franchise_adopt_after = franchise_pair(0, 1, "adopt", "after")

print(f"n = {len(franchise_adopt_after):,}")
print(franchise_adopt_after[["franchise_adopt", "franchise_after"]].isna().mean().rename("share_NA"))
print(pd.crosstab(franchise_adopt_after["franchise_adopt"], franchise_adopt_after["franchise_after"], dropna=False))
franchise_adopt_after.head()

In [ ]:
# =====================================================================
# Franchise transitions: share changing (any / higher / lower) vs. staying
# Denominator = people with a valid (non-NA) franchise in both years
# =====================================================================
def franchise_transitions(df, col_from, col_to, label):
    d = df[[col_from, col_to]].dropna()
    diff = d[col_to] - d[col_from]
    n = len(d)
    return pd.Series({
        "comparison":         label,
        "n_valid":            n,
        "n_excluded_NA":      len(df) - n,
        "pct_changed_any":    (diff != 0).mean() * 100,
        "pct_changed_higher": (diff > 0).mean() * 100,
        "pct_changed_lower":  (diff < 0).mean() * 100,
        "pct_same":           (diff == 0).mean() * 100,
    })

franchise_transition_summary = pd.DataFrame([
    franchise_transitions(franchise_prior_adopt, "franchise_prior", "franchise_adopt",
                          "Year before adoption -> adoption year"),
    franchise_transitions(franchise_adopt_after, "franchise_adopt", "franchise_after",
                          "Adoption year -> year after"),
]).set_index("comparison")

pct_cols = [c for c in franchise_transition_summary.columns if c.startswith("pct_")]
franchise_transition_summary[pct_cols] = franchise_transition_summary[pct_cols].astype(float).round(2)
franchise_transition_summary[["n_valid", "n_excluded_NA"]] = franchise_transition_summary[["n_valid", "n_excluded_NA"]].astype(int)

print(franchise_transition_summary.to_string())
franchise_transition_summary.to_csv(Path(OUT_DIR) / "franchise_transitions_main.csv")

In [ ]:
from scipy.stats import binomtest

d = franchise_adopt_after[["franchise_adopt", "franchise_after"]].dropna()
n_higher = int((d["franchise_after"] > d["franchise_adopt"]).sum())
n_lower  = int((d["franchise_after"] < d["franchise_adopt"]).sum())

result = binomtest(k=n_higher, n=n_higher + n_lower, p=0.5, alternative="two-sided")

print("Proportion of changes that were upward:", n_higher / (n_higher + n_lower))
print("p-value:", result.pvalue)
print("95% CI:", result.proportion_ci())

In [ ]:
# =====================================================================
# Adult franchise distribution by calendar year (main analysis sample)
# =====================================================================
ADULT_FRANCHISES = (300, 500, 1000, 1500, 2000, 2500)
YEAR_MIN, YEAR_MAX = 2018, 2024

franchise_year = con.execute(f"""
    SELECT yr AS year, CAST(franchise AS INTEGER) AS franchise, COUNT(*) AS n_people
    FROM fr_py
    WHERE franchise IN {ADULT_FRANCHISES}
      AND yr BETWEEN {YEAR_MIN} AND {YEAR_MAX}
    GROUP BY 1, 2
    ORDER BY 1, 2
""").df()

plot_df = (franchise_year
           .pivot(index="year", columns="franchise", values="n_people")
           .reindex(index=range(YEAR_MIN, YEAR_MAX + 1), columns=ADULT_FRANCHISES)
           .fillna(0)
           .astype(int))
pct_df = plot_df.div(plot_df.sum(axis=1), axis=0) * 100

print("People per year (adult franchise):")
print(plot_df.sum(axis=1).to_string())

colors = ["#2F4B7C", "#506784", "#748CAB", "#9AA6B2", "#8C6D62", "#5F6F52"]

fig, ax = plt.subplots(figsize=(12, 11))
plot_df.plot(kind="bar", ax=ax, color=colors, width=0.8)

ax.set_xlabel("Calendar year", fontsize=16)
ax.set_ylabel("Number of people", fontsize=16)
ax.set_title("Franchise distribution by calendar year", fontsize=18, pad=14)
ax.tick_params(axis="both", labelsize=14)
ax.tick_params(axis="x", rotation=0)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.spines[["top", "right"]].set_visible(False)
ax.legend(title="Franchise (CHF)", fontsize=13, title_fontsize=14, frameon=False)

for container, fr in zip(ax.containers, plot_df.columns):
    labels = [f"{p:.1f}%" if v > 0 else "" for v, p in zip(plot_df[fr], pct_df[fr])]
    ax.bar_label(container, labels=labels, padding=3, fontsize=10, rotation=90)

ax.set_ylim(0, plot_df.values.max() * 1.15)   # headroom for rotated labels
plt.tight_layout()
fig.savefig(Path(OUT_DIR) / "franchise_distribution_by_year.png", dpi=300, bbox_inches="tight")
fig.savefig(Path(OUT_DIR) / "franchise_distribution_by_year.pdf", bbox_inches="tight")
plt.show()

pct_df.round(2).to_csv(Path(OUT_DIR) / "franchise_distribution_by_year_pct.csv")

In [ ]:
# =====================================================================
# Adult franchise distribution by calendar year (main analysis sample), in %
# =====================================================================
from matplotlib.ticker import PercentFormatter

ADULT_FRANCHISES = (300, 500, 1000, 1500, 2000, 2500)
YEAR_MIN, YEAR_MAX = 2018, 2024

franchise_year = con.execute(f"""
    SELECT yr AS year, CAST(franchise AS INTEGER) AS franchise, COUNT(*) AS n_people
    FROM fr_py
    WHERE franchise IN {ADULT_FRANCHISES}
      AND yr BETWEEN {YEAR_MIN} AND {YEAR_MAX}
    GROUP BY 1, 2
    ORDER BY 1, 2
""").df()

plot_df = (franchise_year
           .pivot(index="year", columns="franchise", values="n_people")
           .reindex(index=range(YEAR_MIN, YEAR_MAX + 1), columns=ADULT_FRANCHISES)
           .fillna(0)
           .astype(int))
n_year = plot_df.sum(axis=1)
pct_df = plot_df.div(n_year, axis=0) * 100

colors = ["#2F4B7C", "#506784", "#748CAB", "#9AA6B2", "#8C6D62", "#5F6F52"]

fig, ax = plt.subplots(figsize=(12, 9))
pct_df.plot(kind="bar", ax=ax, color=colors, width=0.8)

ax.set_xlabel("Calendar year", fontsize=16)
ax.set_ylabel("Share of people (%)", fontsize=16)
ax.set_title("Franchise distribution by calendar year", fontsize=18, pad=14)
ax.tick_params(axis="both", labelsize=14)
ax.set_xticklabels([f"{y}\n(n={n:,})" for y, n in n_year.items()], rotation=0, fontsize=13)
ax.yaxis.set_major_formatter(PercentFormatter(100, decimals=0))
ax.spines[["top", "right"]].set_visible(False)
ax.legend(title="Franchise (CHF)", fontsize=13, title_fontsize=14, frameon=False)

for container, fr in zip(ax.containers, pct_df.columns):
    labels = [f"{p:.1f}%" if p > 0 else "" for p in pct_df[fr]]
    ax.bar_label(container, labels=labels, padding=3, fontsize=10, rotation=90)

ax.set_ylim(0, pct_df.values.max() * 1.15)   # headroom for rotated labels
plt.tight_layout()
fig.savefig(Path(OUT_DIR) / "franchise_distribution_by_year.png", dpi=300, bbox_inches="tight")
plt.show()


## Types of engagment with DHI content

In [ ]:
# =====================================================================
# Reusable: participation in a reward module by year since adoption
# Denominator = main-sample adopters with >= MIN_FOLLOWUP_MONTHS of the relative year observed
# =====================================================================
from matplotlib.ticker import PercentFormatter

def module_participation_by_rel_year(con, patterns, ylabel, file_stem,
                                     exclude_patterns=None, main_ids_df=main_analysis_ids,
                                     max_rel_year=3, min_followup_months=6,
                                     data_end_cap="2024-12-31", out_dir=OUT_DIR,
                                     color="#2F4B7C", show_matches=True):
    """
    patterns          : list of substrings (case-insensitive) matched on reward_module_description
    exclude_patterns  : optional list of substrings to exclude
    ylabel            : y-axis label of the plot
    file_stem         : PNG is saved as <out_dir>/<file_stem>.png
    Returns a DataFrame with rel_year, n_eligible, n_participating, pct_participating.
    """
    col = "reward_module_description"
    where = "(" + " OR ".join(f"{col} ILIKE '%{p}%'" for p in patterns) + ")"
    if exclude_patterns:
        where += " AND NOT (" + " OR ".join(f"{col} ILIKE '%{p}%'" for p in exclude_patterns) + ")"

    if show_matches:
        matches = con.execute(f"""
            SELECT {col}, COUNT(*) AS n_transactions, COUNT(DISTINCT CSS_MARKETING_ID) AS n_people
            FROM a365_transaction_check WHERE {where}
            GROUP BY 1 ORDER BY 2 DESC
        """).df()
        print(f"--- {file_stem}: matched module descriptions ---")
        for _, r in matches.iterrows():
            print(f"[{r.n_transactions:,} tx | {r.n_people:,} people] {r[col]!r}")

    data_max = pd.Timestamp(con.execute(
        "SELECT MAX(TRANSACTION_DATETIME) FROM a365_transaction_check").fetchone()[0])
    data_end = min(data_max.normalize(), pd.Timestamp(data_end_cap))

    ids = main_ids_df[["eth_id"]].astype(str).drop_duplicates().rename(columns={"eth_id": "id"})
    con.register("module_ids_view", ids)

    res = con.execute(f"""
        WITH adopt AS (
            SELECT CAST(CSS_MARKETING_ID AS VARCHAR)       AS id,
                   CAST(MIN(TRANSACTION_DATETIME) AS DATE) AS adopt_d
            FROM a365_transaction_check
            WHERE UPPER(TRIM(TRANSACTION_TYPE)) = 'CREDIT'
              AND CAST(CSS_MARKETING_ID AS VARCHAR) IN (SELECT id FROM module_ids_view)
            GROUP BY 1
        ),
        rel_years AS (
            SELECT a.id, a.adopt_d, k AS rel_year
            FROM adopt a, range(1, {max_rel_year} + 1) r(k)
            WHERE a.adopt_d + INTERVAL (k - 1) YEAR + INTERVAL ({min_followup_months}) MONTH
                  - INTERVAL 1 DAY <= DATE '{data_end.date()}'
        ),
        hits AS (
            SELECT DISTINCT CAST(t.CSS_MARKETING_ID AS VARCHAR) AS id,
                   FLOOR(DATE_DIFF('day', a.adopt_d, CAST(t.TRANSACTION_DATETIME AS DATE)) / 365.25) + 1 AS rel_year
            FROM a365_transaction_check t
            JOIN adopt a ON a.id = CAST(t.CSS_MARKETING_ID AS VARCHAR)
            WHERE {where.replace(col, 't.' + col)}
              AND CAST(t.TRANSACTION_DATETIME AS DATE) >= a.adopt_d
        )
        SELECT r.rel_year,
               COUNT(*)                        AS n_eligible,
               COUNT(h.id)                     AS n_participating,
               100.0 * COUNT(h.id) / COUNT(*)  AS pct_participating
        FROM rel_years r
        LEFT JOIN hits h ON h.id = r.id AND h.rel_year = r.rel_year
        GROUP BY 1 ORDER BY 1
    """).df()
    con.unregister("module_ids_view")
    print(res.to_string(index=False), "\n")

    fig, ax = plt.subplots(figsize=(8, 6))
    bars = ax.bar(range(len(res)), res["pct_participating"], color=color, width=0.6)
    ax.bar_label(bars, labels=[f"{n:,}\n({p:.1f}%)" for n, p in
                               zip(res["n_participating"], res["pct_participating"])],
                 padding=4, fontsize=12)
    ax.set_xticks(range(len(res)))
    ax.set_xticklabels([f"Year {int(k)}\n(n={n:,})" for k, n in
                        zip(res["rel_year"], res["n_eligible"])], fontsize=12)
    ax.set_ylabel(ylabel, fontsize=13)
    ax.set_xlabel("Year since adoption", fontsize=13)
    ax.yaxis.set_major_formatter(PercentFormatter(100, decimals=0))
    ax.set_ylim(0, 25)
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    fig.savefig(Path(out_dir) / f"{file_stem}.png", dpi=300, bbox_inches="tight")
    plt.show()
    return res

In [ ]:
MODULES = {
    "medical_checkup_module_by_rel_year": dict(
        patterns=["medical check-ups"],
        ylabel="Adopters with preventive medical check-ups linked (%)"),
    "sports_club_module_by_rel_year": dict(
        patterns=["sports club"],
        ylabel="Adopters with sports-club module activity (%)"),
    "preventive_checkup_module_by_rel_year": dict(
        patterns=["preventive check-ups"],
        ylabel="Adopters with preventive check-ups linked (%)"),
}

module_results = {}
for stem, cfg in MODULES.items():
    module_results[stem] = module_participation_by_rel_year(con, file_stem=stem, **cfg)

# Combined overview table
module_summary = pd.concat(module_results, names=["module"]).reset_index(level=0)
module_summary

In [ ]:
# =====================================================================
# Distinct health session types
# =====================================================================
pd.set_option("display.max_colwidth", None)

sess_cols = con.execute("DESCRIBE a365_health_sessions").df()
print(sess_cols[["column_name", "column_type"]].to_string(index=False), "\n")

type_cols = [c for c in sess_cols["column_name"] if "TYPE" in c.upper()]
SESS_TYPE_COL = type_cols[0] if type_cols else "SESSION_MAIN_CATEGORY"
print("Grouping by:", SESS_TYPE_COL)

session_types = con.execute(f"""
    SELECT {SESS_TYPE_COL}                   AS session_type,
           COUNT(*)                          AS n_sessions,
           COUNT(DISTINCT CSS_MARKETING_ID)  AS n_people,
           MIN(DATE_TIME)                    AS first_seen,
           MAX(DATE_TIME)                    AS last_seen
    FROM a365_health_sessions
    GROUP BY 1
    ORDER BY n_sessions DESC
""").df()

print(f"{len(session_types):,} distinct values")
session_types

In [ ]:
# =====================================================================
# Health sessions by main category (mind / move / nut)
#  Chart 1: participation in relative year 1 and 2 after adoption
#           (only person-years in which sessions were available, see SESSION_RULE)
#  Chart 2: participation in calendar years 2023 and 2024
#           (people who had adopted by the end of that year)
# =====================================================================
from matplotlib.ticker import PercentFormatter

SESSIONS_START = "2022-12-31"
SESSION_RULE = "overlap"   # "overlap": sessions available at some point in year k
                           # "full"   : sessions available for all of year k (stricter)
MIN_FOLLOWUP_MONTHS = 6

CATEGORIES = ["mind", "move", "nut"]
CAT_COLORS = {"mind": "#2F4B7C", "move": "#9AA6B2", "nut": "#5F6F52"}

sess_cols = con.execute("DESCRIBE a365_health_sessions").df()["column_name"].tolist()
cat_cands = [c for c in sess_cols if "MAIN" in c.upper() and "CATEG" in c.upper()]
if not cat_cands:
    raise ValueError(f"No main-category column found: {sess_cols}")
CAT_COL = cat_cands[0]
print("Category column:", CAT_COL)

data_max = pd.Timestamp(con.execute(
    "SELECT MAX(TRANSACTION_DATETIME) FROM a365_transaction_check").fetchone()[0])
DATA_END = min(data_max.normalize(), pd.Timestamp("2024-12-31"))

ids = main_analysis_ids[["eth_id"]].astype(str).drop_duplicates().rename(columns={"eth_id": "id"})
con.register("sess_ids_view", ids)

cat_case = "CASE " + " ".join(
    f"WHEN LOWER(TRIM({CAT_COL})) LIKE '{c}%' THEN '{c}'" for c in CATEGORIES) + " END"

con.execute("""
    CREATE OR REPLACE TEMP TABLE sess_adopt AS
    SELECT CAST(CSS_MARKETING_ID AS VARCHAR)       AS id,
           CAST(MIN(TRANSACTION_DATETIME) AS DATE) AS adopt_d
    FROM a365_transaction_check
    WHERE UPPER(TRIM(TRANSACTION_TYPE)) = 'CREDIT'
      AND CAST(CSS_MARKETING_ID AS VARCHAR) IN (SELECT id FROM sess_ids_view)
    GROUP BY 1
""")
con.execute(f"""
    CREATE OR REPLACE TEMP TABLE sess_main AS
    SELECT DISTINCT CAST(s.CSS_MARKETING_ID AS VARCHAR) AS id,
           CAST(s.DATE_TIME AS DATE)                    AS d,
           {cat_case}                                   AS cat
    FROM a365_health_sessions s
    WHERE CAST(s.CSS_MARKETING_ID AS VARCHAR) IN (SELECT id FROM sess_adopt)
      AND ({cat_case}) IS NOT NULL
""")
print(con.execute("SELECT cat, COUNT(*) AS n_person_days, COUNT(DISTINCT id) AS n_people "
                  "FROM sess_main GROUP BY 1 ORDER BY 1").df().to_string(index=False), "\n")

cats_sql = ", ".join(f"('{c}')" for c in CATEGORIES)

if SESSION_RULE == "overlap":
    avail_cond = f"a.adopt_d + INTERVAL (k) YEAR > DATE '{SESSIONS_START}'"
elif SESSION_RULE == "full":
    avail_cond = f"a.adopt_d + INTERVAL (k - 1) YEAR >= DATE '{SESSIONS_START}'"
else:
    raise ValueError("SESSION_RULE must be 'overlap' or 'full'")

# ---- 1) Relative years 1 and 2 after adoption ------------------------
rel = con.execute(f"""
    WITH elig AS (
        SELECT a.id, a.adopt_d, k AS period
        FROM sess_adopt a, range(1, 3) r(k)
        WHERE a.adopt_d + INTERVAL (k - 1) YEAR + INTERVAL ({MIN_FOLLOWUP_MONTHS}) MONTH
              - INTERVAL 1 DAY <= DATE '{DATA_END.date()}'
          AND {avail_cond}
    ),
    part AS (
        SELECT DISTINCT s.id, s.cat,
               FLOOR(DATE_DIFF('day', a.adopt_d, s.d) / 365.25) + 1 AS period
        FROM sess_main s JOIN sess_adopt a USING (id)
        WHERE s.d >= a.adopt_d
    ),
    grid AS (SELECT e.id, e.period, c.cat FROM elig e, (VALUES {cats_sql}) c(cat))
    SELECT g.period, g.cat,
           COUNT(*)                        AS n_eligible,
           COUNT(p.id)                     AS n_participating,
           100.0 * COUNT(p.id) / COUNT(*)  AS pct_participating
    FROM grid g
    LEFT JOIN part p ON p.id = g.id AND p.period = g.period AND p.cat = g.cat
    GROUP BY 1, 2 ORDER BY 1, 2
""").df()
print(f"Relative years (SESSION_RULE = '{SESSION_RULE}')")
print(rel.to_string(index=False), "\n")

# ---- 2) Calendar years 2023 and 2024 --------------------------------
cal = con.execute(f"""
    WITH elig AS (
        SELECT a.id, y AS period
        FROM sess_adopt a, (VALUES (2023), (2024)) yrs(y)
        WHERE a.adopt_d <= MAKE_DATE(y, 12, 31)
    ),
    part AS (SELECT DISTINCT id, cat, YEAR(d) AS period FROM sess_main),
    grid AS (SELECT e.id, e.period, c.cat FROM elig e, (VALUES {cats_sql}) c(cat))
    SELECT g.period, g.cat,
           COUNT(*)                        AS n_eligible,
           COUNT(p.id)                     AS n_participating,
           100.0 * COUNT(p.id) / COUNT(*)  AS pct_participating
    FROM grid g
    LEFT JOIN part p ON p.id = g.id AND p.period = g.period AND p.cat = g.cat
    GROUP BY 1, 2 ORDER BY 1, 2
""").df()
print("Calendar years")
print(cal.to_string(index=False))

# ---- Plotting --------------------------------------------------------
def grouped_bar(df, period_label, xlabel, file_stem):
    periods = sorted(df["period"].unique())
    x = np.arange(len(periods))
    width = 0.8 / len(CATEGORIES)
    fig, ax = plt.subplots(figsize=(10, 7))
    for i, c in enumerate(CATEGORIES):
        d = df[df["cat"] == c].set_index("period").reindex(periods)
        bars = ax.bar(x + (i - (len(CATEGORIES) - 1) / 2) * width, d["pct_participating"],
                      width=width, color=CAT_COLORS[c], label=c.capitalize())
        ax.bar_label(bars, labels=[f"{n:,}\n({p:.1f}%)" for n, p in
                                   zip(d["n_participating"], d["pct_participating"])],
                     padding=3, fontsize=10)
    n_elig = df.groupby("period")["n_eligible"].first().reindex(periods)
    ax.set_xticks(x)
    ax.set_xticklabels([f"{period_label(p)}\n(n={n:,})" for p, n in zip(periods, n_elig)], fontsize=12)
    ax.set_xlabel(xlabel, fontsize=13)
    ax.set_ylabel("Adopters with ≥1 session (%)", fontsize=13)
    ax.yaxis.set_major_formatter(PercentFormatter(100, decimals=0))
    ax.set_ylim(0, max(df["pct_participating"].max(), 1) * 1.3)
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(title="Session category", frameon=False, fontsize=12, title_fontsize=12,
              loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=len(CATEGORIES))
    plt.tight_layout()
    fig.savefig(Path(OUT_DIR) / f"{file_stem}.png", dpi=300, bbox_inches="tight")
    plt.show()

grouped_bar(rel, lambda p: f"Year {int(p)}", "Year since adoption",
            "health_sessions_by_category_rel_year")
grouped_bar(cal, lambda p: str(int(p)), "Calendar year",
            "health_sessions_by_category_calendar_year")

In [ ]:
# =====================================================================
# Sessions per person by category and calendar year (people with >=1 session)
# Violin plot per year; annotated with mean, median, min, max
# Requires sess_adopt and CAT_COL / CATEGORIES / CAT_COLORS from the session block
# =====================================================================
YEARS = [2023, 2024]
LOG_Y = True

cat_case_raw = "CASE " + " ".join(
    f"WHEN LOWER(TRIM(s.{CAT_COL})) LIKE '{c}%' THEN '{c}'" for c in CATEGORIES) + " END"
title_col = "SESSION_TITLE" if "SESSION_TITLE" in sess_cols else None
session_key = (f"CAST(s.DATE_TIME AS VARCHAR) || '|' || COALESCE(CAST(s.{title_col} AS VARCHAR), '')"
               if title_col else "CAST(s.DATE_TIME AS VARCHAR)")

sess_counts = con.execute(f"""
    SELECT CAST(s.CSS_MARKETING_ID AS VARCHAR) AS id,
           YEAR(s.DATE_TIME)                   AS year,
           {cat_case_raw}                      AS cat,
           COUNT(DISTINCT {session_key})       AS n_sessions
    FROM a365_health_sessions s
    WHERE CAST(s.CSS_MARKETING_ID AS VARCHAR) IN (SELECT id FROM sess_adopt)
      AND YEAR(s.DATE_TIME) IN ({", ".join(map(str, YEARS))})
      AND ({cat_case_raw}) IS NOT NULL
    GROUP BY 1, 2, 3
""").df()

sess_stats = (sess_counts.groupby(["year", "cat"])["n_sessions"]
              .agg(n_people="size", mean="mean", median="median", min="min", max="max")
              .reset_index())
print(sess_stats.round(2).to_string(index=False))

def session_violin(year):
    d = sess_counts[sess_counts["year"] == year]
    data = [d.loc[d["cat"] == c, "n_sessions"].values for c in CATEGORIES]
    pos = np.arange(1, len(CATEGORIES) + 1)

    fig, ax = plt.subplots(figsize=(10, 7))
    plot_data = [np.log10(x) for x in data] if LOG_Y else data
    parts = ax.violinplot(plot_data, positions=pos, widths=0.8,
                          showmeans=False, showmedians=False, showextrema=False)
    for body, c in zip(parts["bodies"], CATEGORIES):
        body.set_facecolor(CAT_COLORS[c]); body.set_edgecolor(CAT_COLORS[c]); body.set_alpha(0.75)

    tr = (lambda v: np.log10(v)) if LOG_Y else (lambda v: v)
    top = max(tr(x.max()) for x in data if len(x))
    for p, x in zip(pos, data):
        if not len(x):
            continue
        ax.hlines(tr(np.median(x)), p - 0.25, p + 0.25, color="white", lw=2.2)        # median
        ax.scatter(p, tr(x.mean()), color="black", s=25, zorder=3)                  # mean
        ax.vlines(p, tr(x.min()), tr(x.max()), color="black", lw=0.8, alpha=0.6)    # range
        ax.text(p, top * 1.08 if LOG_Y else top * 1.05,
                f"n = {len(x):,}\nMean: {x.mean():.1f}\nMedian: {np.median(x):.0f}\n"
                f"Min: {x.min():,}\nMax: {x.max():,}",
                ha="center", va="bottom", fontsize=10)

    ax.set_xticks(pos)
    ax.set_xticklabels([c.capitalize() for c in CATEGORIES], fontsize=13)
    ax.set_xlabel("Session category", fontsize=13)
    ax.set_ylabel("Sessions per person" + (" (log scale)" if LOG_Y else ""), fontsize=13)
    if LOG_Y:
        ticks = [t for t in [1, 2, 5, 10, 20, 50, 100, 200, 500, 1000, 2000, 5000]
                 if t <= 10 ** top * 1.5]
        ax.set_yticks(np.log10(ticks))
        ax.set_yticklabels([f"{t:,}" for t in ticks])
        ax.set_ylim(-0.05, top * 1.45)
    else:
        ax.set_ylim(0, top * 1.35)
    ax.set_title(f"Sessions per person, {year} (people with ≥1 session)", fontsize=15, pad=12)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    fig.savefig(Path(OUT_DIR) / f"sessions_per_person_violin_{year}.png", dpi=300, bbox_inches="tight")
    plt.show()

for y in YEARS:
    session_violin(y)

sess_stats.to_csv(Path(OUT_DIR) / "sessions_per_person_stats.csv", index=False)

In [ ]:
# =====================================================================
# Sessions per person by category and calendar year (people with >=1 session)
# Sessions followed by another session of the same user within MIN_GAP_S
# seconds are dropped. Violin per year, annotated with mean/median/min/max.
# Requires sess_adopt, CAT_COL, CATEGORIES, CAT_COLORS from the session block
# =====================================================================
YEARS = [2023, 2024]
MIN_GAP_S = 60
GAP_WITHIN_CATEGORY = False   # True: only compare to the next session of the same category
LOG_Y = True

cat_case_raw = "CASE " + " ".join(
    f"WHEN LOWER(TRIM(s.{CAT_COL})) LIKE '{c}%' THEN '{c}'" for c in CATEGORIES) + " END"
part_by = "id, cat" if GAP_WITHIN_CATEGORY else "id"

# ---- Clean sessions --------------------------------------------------
con.execute(f"""
    CREATE OR REPLACE TEMP TABLE sess_clean AS
    WITH s AS (
        SELECT DISTINCT
               CAST(s.CSS_MARKETING_ID AS VARCHAR) AS id,
               s.DATE_TIME                         AS ts,
               s.SESSION_TITLE                     AS title,
               {cat_case_raw}                      AS cat
        FROM a365_health_sessions s
        WHERE CAST(s.CSS_MARKETING_ID AS VARCHAR) IN (SELECT id FROM sess_adopt)
          AND ({cat_case_raw}) IS NOT NULL
    ),
    g AS (
        SELECT *,
               DATE_DIFF('second', ts, LEAD(ts) OVER (PARTITION BY {part_by} ORDER BY ts)) AS gap_next_s
        FROM s
    )
    SELECT *, (gap_next_s IS NULL OR gap_next_s >= {MIN_GAP_S}) AS keep
    FROM g
""")

removal = con.execute(f"""
    SELECT YEAR(ts) AS year, cat,
           COUNT(*)                          AS n_before,
           SUM(CASE WHEN keep THEN 1 ELSE 0 END)     AS n_after,
           SUM(CASE WHEN NOT keep THEN 1 ELSE 0 END) AS n_removed
    FROM sess_clean
    WHERE YEAR(ts) IN ({", ".join(map(str, YEARS))})
    GROUP BY 1, 2 ORDER BY 1, 2
""").df()
removal["pct_removed"] = 100 * removal["n_removed"] / removal["n_before"]
print(f"Sessions removed (followed by another session within {MIN_GAP_S}s):")
print(removal.round(2).to_string(index=False), "\n")

# ---- Counts per person ----------------------------------------------
sess_counts = con.execute(f"""
    SELECT id, YEAR(ts) AS year, cat, COUNT(*) AS n_sessions
    FROM sess_clean
    WHERE keep AND YEAR(ts) IN ({", ".join(map(str, YEARS))})
    GROUP BY 1, 2, 3
""").df()

sess_stats = (sess_counts.groupby(["year", "cat"])["n_sessions"]
              .agg(n_people="size", mean="mean", median="median",
                   q25=lambda s: s.quantile(0.25), q75=lambda s: s.quantile(0.75),
                   min="min", max="max")
              .reset_index())
print(sess_stats.round(2).to_string(index=False))

# ---- Plot ------------------------------------------------------------
def session_violin(year):
    d = sess_counts[sess_counts["year"] == year]
    data = [d.loc[d["cat"] == c, "n_sessions"].values for c in CATEGORIES]
    pos = np.arange(1, len(CATEGORIES) + 1)
    tr = np.log10 if LOG_Y else (lambda v: v)

    fig, ax = plt.subplots(figsize=(10, 7))
    parts = ax.violinplot([tr(x) for x in data], positions=pos, widths=0.8,
                          showmeans=False, showmedians=False, showextrema=False)
    for body, c in zip(parts["bodies"], CATEGORIES):
        body.set_facecolor(CAT_COLORS[c]); body.set_edgecolor(CAT_COLORS[c]); body.set_alpha(0.75)

    top = max(tr(x.max()) for x in data if len(x))
    for p, x in zip(pos, data):
        if not len(x):
            continue
        ax.vlines(p, tr(x.min()), tr(x.max()), color="black", lw=0.8, alpha=0.6)   # range
        ax.hlines(tr(np.median(x)), p - 0.25, p + 0.25, color="white", lw=2.2)      # median
        ax.scatter(p, tr(x.mean()), color="black", s=25, zorder=3)                 # mean
        ax.text(p, top * 1.06 if LOG_Y else top * 1.04,
                f"n = {len(x):,}\nMean: {x.mean():.1f}\nMedian: {np.median(x):.0f}\n"
                f"Min: {x.min():,}\nMax: {x.max():,}",
                ha="center", va="bottom", fontsize=10)

    ax.set_xticks(pos)
    ax.set_xticklabels([c.capitalize() for c in CATEGORIES], fontsize=13)
    ax.set_xlabel("Session category", fontsize=13)
    ax.set_ylabel("Sessions per person" + (" (log scale)" if LOG_Y else ""), fontsize=13)
    if LOG_Y:
        ticks = [t for t in [1, 2, 5, 10, 20, 50, 100, 200, 500, 1000, 2000, 5000]
                 if t <= 10 ** top * 1.5]
        ax.set_yticks(np.log10(ticks))
        ax.set_yticklabels([f"{t:,}" for t in ticks])
        ax.set_ylim(-0.05, top * 1.45)
    else:
        ax.set_ylim(0, top * 1.35)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    fig.savefig(Path(OUT_DIR) / f"sessions_per_person_violin_{year}_clean.png", dpi=300, bbox_inches="tight")
    plt.show()

for y in YEARS:
    session_violin(y)

sess_stats.to_csv(Path(OUT_DIR) / "sessions_per_person_stats_clean.csv", index=False)
removal.to_csv(Path(OUT_DIR) / "sessions_removed_min_gap.csv", index=False)

In [ ]:
tot = removal.groupby("year")[["n_before", "n_removed"]].sum()
tot.loc["Total"] = tot.sum()
tot["pct_removed"] = (100 * tot["n_removed"] / tot["n_before"]).round(2)
tot

## Timing of chronic diagnoses (PCGs)

In [ ]:
# =====================================================================
# First-time PCGs relative to DHI adoption (main analysis sample)
#  - adoption year  = YEAR(start_month) from the analysis panel
#                     (start_month = month of first transaction of any type)
#  - observed years = calendar years with >=1 panel month
#                     (panel months = active persons spells, as in main-sample construction)
#  - one row per person x pcg_jahr x PCG (deduplicated across ra_jahr / versions)
#  - each PCG counted only in the first pcg_jahr it appears for that person
#  Figure 1: number of first-time PCG assignments by year relative to adoption
#  Figure 2: share of observed persons with >=1 first-time PCG
# =====================================================================
from matplotlib.ticker import PercentFormatter

PANEL = "claims_person_month_summary_final_w_start_engagementq"

# ---- PCG columns -----------------------------------------------------
pcg_desc = con.execute("DESCRIBE pcg_long").df()
print(pcg_desc[["column_name", "column_type"]].to_string(index=False), "\n")

skip = {"eth_id", "ra_jahr", "pcg_jahr", "pcg_versionsnummer"}
str_cols = [r.column_name for r in pcg_desc.itertuples()
            if r.column_name not in skip and r.column_type == "VARCHAR"]
num_cols = [r.column_name for r in pcg_desc.itertuples()
            if r.column_name not in skip and r.column_type in ("BIGINT", "INTEGER", "DOUBLE", "HUGEINT")]
PCG_COL = next((c for c in str_cols if "pcg" in c.lower()), str_cols[0] if str_cols else None)
PCG_FLAG_COL = num_cols[0] if num_cols else None      # set to None if it is not a 0/1 indicator
if PCG_COL is None:
    raise ValueError("Could not identify the PCG column -> set PCG_COL manually")
print(f"PCG identifier column: {PCG_COL} | indicator column: {PCG_FLAG_COL}")
flag_cond = f"AND {PCG_FLAG_COL} > 0" if PCG_FLAG_COL else ""

ids = main_analysis_ids[["eth_id"]].astype(str).drop_duplicates().rename(columns={"eth_id": "id"})
con.register("pcg_ids_view", ids)

# ---- 1) Adoption year from the analysis panel ------------------------
con.execute(f"""
    CREATE OR REPLACE TEMP TABLE pcg_adopt AS
    SELECT CAST(eth_id AS VARCHAR) AS id,
           YEAR(MIN(start_month))  AS adopt_year
    FROM {PANEL}
    WHERE start_month IS NOT NULL
      AND CAST(eth_id AS VARCHAR) IN (SELECT id FROM pcg_ids_view)
    GROUP BY 1
""")
n_main = con.execute("SELECT COUNT(*) FROM pcg_adopt").fetchone()[0]
print(f"Main-sample persons with adoption date in panel: {n_main:,} (expected {len(ids):,})")

# ---- 2) First occurrence of each PCG per person ----------------------
con.execute(f"""
    CREATE OR REPLACE TEMP TABLE pcg_first AS
    WITH uniq AS (
        SELECT DISTINCT CAST(eth_id AS VARCHAR) AS id,
               CAST(pcg_jahr AS INTEGER)       AS pcg_jahr,
               CAST({PCG_COL} AS VARCHAR)      AS pcg
        FROM pcg_long
        WHERE CAST(eth_id AS VARCHAR) IN (SELECT id FROM pcg_adopt)
          AND {PCG_COL} IS NOT NULL
          {flag_cond}
    )
    SELECT u.id, u.pcg, MIN(u.pcg_jahr) AS first_pcg_jahr, a.adopt_year,
           MIN(u.pcg_jahr) - a.adopt_year AS rel_year
    FROM uniq u JOIN pcg_adopt a USING (id)
    GROUP BY u.id, u.pcg, a.adopt_year
""")

yr_min, yr_max = con.execute("SELECT MIN(pcg_jahr), MAX(pcg_jahr) FROM pcg_long").fetchone()
print(f"pcg_jahr range in data: {yr_min}–{yr_max}")

# ---- 3) Observed calendar years = years with >=1 panel month ---------
con.execute(f"""
    CREATE OR REPLACE TEMP TABLE pcg_observed AS
    SELECT DISTINCT CAST(p.eth_id AS VARCHAR) AS id,
           a.adopt_year,
           YEAR(p.claim_month)                AS cal_year,
           YEAR(p.claim_month) - a.adopt_year AS rel_year
    FROM {PANEL} p
    JOIN pcg_adopt a ON a.id = CAST(p.eth_id AS VARCHAR)
""")

chk = con.execute("SELECT rel_year, COUNT(DISTINCT id) AS n FROM pcg_observed "
                  "WHERE rel_year IN (-1, 0) GROUP BY 1 ORDER BY 1").df()
for r in chk.itertuples():
    print(f"Observed in year {r.rel_year:+d}: {r.n:,} of {n_main:,} ({r.n / n_main:.1%})")

# ---- 4) Tables -------------------------------------------------------
pcg_rel = con.execute("""
    SELECT rel_year,
           COUNT(*)           AS n_new_pcgs,
           COUNT(DISTINCT id) AS n_people_with_new_pcg
    FROM pcg_first
    GROUP BY 1 ORDER BY 1
""").df()
print("\nFirst-time PCG assignments (all years):")
print(pcg_rel.to_string(index=False))

pcg_rate = con.execute("""
    SELECT o.rel_year,
           COUNT(DISTINCT o.id)                                 AS n_observed,
           COUNT(f.pcg)                                         AS n_new_pcgs,
           COUNT(DISTINCT f.id)                                 AS n_people_with_new_pcg,
           100.0 * COUNT(DISTINCT f.id) / COUNT(DISTINCT o.id)  AS pct_people_with_new_pcg
    FROM pcg_observed o
    LEFT JOIN pcg_first f ON f.id = o.id AND f.rel_year = o.rel_year
    GROUP BY 1 ORDER BY 1
""").df()
print("\nShare of observed persons with >=1 first-time PCG:")
print(pcg_rate.round(2).to_string(index=False))

# ---- Figure 1: number of first-time PCG assignments ------------------
fig, ax = plt.subplots(figsize=(11, 6))

colors_bar = ["#D1495B" if r == 0 else "#2F4B7C"
              for r in pcg_rel["rel_year"]]

bars = ax.bar(
    pcg_rel["rel_year"],
    pcg_rel["n_new_pcgs"],
    color=colors_bar,
    width=0.75
)

ax.bar_label(
    bars,
    labels=[fmt_int_lancet(v) for v in pcg_rel["n_new_pcgs"]],
    padding=3,
    fontsize=10
)

ax.set_xticks(pcg_rel["rel_year"])
ax.set_xticklabels(
    [f"{r:+d}" if r != 0 else "0" for r in pcg_rel["rel_year"]],
    fontsize=12
)

ax.set_xlabel("Year relative to DHI adoption", fontsize=13)
ax.set_ylabel("Number of first-time PCG assignments", fontsize=13)

ax.yaxis.set_major_formatter(
    plt.FuncFormatter(lambda x, _: fmt_int_lancet(x))
)

ax.set_ylim(0, pcg_rel["n_new_pcgs"].max() * 1.12)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()

fig.savefig(
    Path(OUT_DIR) / "pcg_first_occurrence_by_rel_year.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ---- Figure 2: share of observed persons with >=1 first-time PCG -----
fig, ax = plt.subplots(figsize=(14, 7))

colors_bar = ["#D1495B" if r == 0 else "#2F4B7C"
              for r in pcg_rate["rel_year"]]
bars = ax.bar(
    pcg_rate["rel_year"],
    pcg_rate["pct_people_with_new_pcg"],
    color=colors_bar,
    width=0.75
)
ax.bar_label(
    bars,
    labels=[
        fmt_pct_lancet(v, decimals=1)
        for v in pcg_rate["pct_people_with_new_pcg"]
    ],
    padding=3,
    fontsize=10
)

ax.set_xticks(pcg_rate["rel_year"])

ax.set_xticklabels(
    [
        (
            f"{'0' if r == 0 else f'{r:+d}'}\n"
            f"obs. n={fmt_int_lancet(n_obs)}\n"
            f"PCG n={fmt_int_lancet(n_pcg)}"
        )
        for r, n_obs, n_pcg in zip(
            pcg_rate["rel_year"],
            pcg_rate["n_observed"],
            pcg_rate["n_people_with_new_pcg"]
        )
    ],
    fontsize=9
)

ax.set_xlabel(
    "Year relative to DHI adoption",
    fontsize=13,
    labelpad=8
)

ax.set_ylabel(
    "Observed persons with ≥1 first-time PCG (%)",
    fontsize=13
)

# Lancet-style centred decimal dot if non-integer percentage ticks occur
ax.yaxis.set_major_formatter(
    plt.FuncFormatter(
        lambda x, _: (
            fmt_int_lancet(x) + "%"
            if float(x).is_integer()
            else fmt_pct_lancet(x, decimals=1)
        )
    )
)

ax.set_ylim(
    0,
    pcg_rate["pct_people_with_new_pcg"].max() * 1.15
)

ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()

fig.savefig(
    Path(OUT_DIR) / "pcg_first_occurrence_pct_by_rel_year.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

pcg_rel.to_csv(
    Path(OUT_DIR) / "pcg_first_occurrence_by_rel_year.csv",
    index=False
)

pcg_rate.to_csv(
    Path(OUT_DIR) / "pcg_first_occurrence_rate_by_rel_year.csv",
    index=False
)

## Reward points paid out after adoption


In [ ]:
# =====================================================================
# activePoints paid out after adoption (main analysis sample)
#  - adoption = first transaction of any type (same as start_month in the DiD panel)
#  - window   = adoption quarter + following quarters, POST_QUARTERS in total (6 = 1.5 years)
#  - points   = sum of CREDIT transactions in the window
#  - summaries only use people whose full window is observed by DATA_END
# =====================================================================

POST_QUARTERS = 6
CHF_PER_POINT = 10

last_full_q = DATA_END.year * 4 + DATA_END.quarter - 1 - (0 if DATA_END.is_quarter_end else 1)

ids = main_analysis_ids[["eth_id"]].astype(str).drop_duplicates().rename(columns={"eth_id": "id"})
con.register("pts_ids_view", ids)

con.execute(f"""
    CREATE OR REPLACE TEMP TABLE pts_person AS
    WITH a AS (
        SELECT CAST(CSS_MARKETING_ID AS VARCHAR)       AS id,
               CAST(MIN(TRANSACTION_DATETIME) AS DATE) AS adopt_d
        FROM a365_transaction_check
        WHERE TRANSACTION_DATETIME IS NOT NULL
          AND CAST(CSS_MARKETING_ID AS VARCHAR) IN (SELECT id FROM pts_ids_view)
        GROUP BY 1
    ),
    g AS (
        SELECT id, adopt_d, YEAR(adopt_d) * 4 + QUARTER(adopt_d) - 1 AS g_q FROM a
    ),
    q AS (
        SELECT CAST(t.CSS_MARKETING_ID AS VARCHAR)                                      AS id,
               YEAR(t.TRANSACTION_DATETIME) * 4 + QUARTER(t.TRANSACTION_DATETIME) - 1  AS t_q,
               SUM(t.TRANSACTION_VALUE)                                                 AS points
        FROM a365_transaction_check t
        JOIN g ON g.id = CAST(t.CSS_MARKETING_ID AS VARCHAR)
        WHERE UPPER(TRIM(t.TRANSACTION_TYPE)) = 'CREDIT'
          AND CAST(t.TRANSACTION_DATETIME AS DATE) <= DATE '{DATA_END.date()}'
        GROUP BY 1, 2
    )
    SELECT g.id                                         AS eth_id,
           g.adopt_d,
           YEAR(g.adopt_d)                              AS adopt_year,
           COALESCE(SUM(q.points), 0)                   AS total_credit_points,
           COALESCE(SUM(q.points), 0) / {POST_QUARTERS} AS avg_quarterly_credit_points,
           g.g_q + {POST_QUARTERS} - 1 <= {last_full_q} AS fully_observed
    FROM g
    LEFT JOIN q ON q.id = g.id AND q.t_q BETWEEN g.g_q AND g.g_q + {POST_QUARTERS} - 1
    GROUP BY g.id, g.adopt_d, g.g_q
""")

pts = con.execute("SELECT * FROM pts_person").df()
if CHF_PER_POINT:
    pts["total_chf"] = pts["total_credit_points"] * CHF_PER_POINT
    pts["avg_quarterly_chf"] = pts["avg_quarterly_credit_points"] * CHF_PER_POINT

full = pts[pts["fully_observed"]]
print(f"Adopters: {len(pts):,} | full {POST_QUARTERS}-quarter window observed: {len(full):,} "
      f"({100 * len(full) / len(pts):.1f}%)\n")

val_cols = [c for c in ["total_credit_points", "avg_quarterly_credit_points",
                        "total_chf", "avg_quarterly_chf"] if c in full.columns]
print(full[val_cols].describe(percentiles=[.25, .5, .75]).T.round(1).to_string(), "\n")

agg = {"n": ("eth_id", "size"),
       "total_points": ("total_credit_points", "sum"),
       "mean_quarterly_points": ("avg_quarterly_credit_points", "mean"),
       "median_quarterly_points": ("avg_quarterly_credit_points", "median")}
if CHF_PER_POINT:
    agg["total_chf"] = ("total_chf", "sum")
by_year = full.groupby("adopt_year").agg(**agg)
total = by_year.sum()
total[["mean_quarterly_points", "median_quarterly_points"]] = [
    full["avg_quarterly_credit_points"].mean(), full["avg_quarterly_credit_points"].median()]
by_year.loc["Total"] = total
print(by_year.round(1).to_string())

by_year.round(2).to_csv(Path(OUT_DIR) / "credit_points_post_adoption_by_cohort.csv")

pts.to_csv(Path(OUT_DIR) / "credit_points_post_adoption_person.csv", index=False)

In [ ]:
con.close()